**Topic:** Climate Change

Install necessary libraries

In [36]:
!pip install -q wikipedia-api
!pip install -q chromadb
!pip install -q sentence_transformers
!pip install -q openai

In [22]:
import wikipediaapi
import chromadb
from sentence_transformers import SentenceTransformer

Select five Wikipedia pages

In [23]:
wiki = wikipediaapi.Wikipedia(user_agent="RAG_Assignment/1.0 (soojalrathore708@gmail.com)", language="en")
topics = [
    "Climate change",
    "Greenhouse gas",
    "Effects of climate change",
    "Climate change mitigation",
    "Climate change adaptation",
]

pages = [wiki.page(t) for t in topics]
for p in pages:
    print(p.title, "->", p.exists())

Climate change -> True
Greenhouse gas -> True
Effects of climate change -> True
Climate change mitigation -> True
Climate change adaptation -> True


Extract and store paragraphs

In [24]:
MIN_LENGTH = 150

documents = []
metadatas = []
for p in pages:
    for para in p.text.split("\n"):
        para = para.strip()
        if len(para) >= MIN_LENGTH and not para.endswith(":"):
            documents.append(para)
            metadatas.append({"source": p.title})

print("Total paragraphs:", len(documents))
documents[0]

Total paragraphs: 507


"Present-day climate change includes both global warming—the ongoing increase in global average temperature—and its wider effects on Earth's climate system. In a broader sense, climate change also includes previous long-term changes to Earth's climate. The modern-day rise in global temperatures is driven by human activities, especially fossil fuel (coal, oil and natural gas) burning since the Industrial Revolution. Fossil fuel use, deforestation, and some agricultural and industrial practices release greenhouse gases. These gases absorb some of the heat that the Earth radiates after it warms from sunlight, warming the lower atmosphere. Earth's atmosphere now has roughly 50% more carbon dioxide, the main gas driving global warming, than it did at the end of the pre-industrial era, reaching levels not seen for millions of years."

Convert text into vector embeddings (sentence-transformers)

In [25]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(507, 384)


Store the embeddings in ChromaDB

In [26]:
chroma_client = chromadb.PersistentClient(path="/content/chromadb")

In [27]:
collection_name = "wikipedia_climate_collection"

In [28]:
if collection_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(name=collection_name)
collection = chroma_client.create_collection(name=collection_name)

In [29]:
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
    ids=[f"id{x}" for x in range(len(documents))],
)
print("Stored in ChromaDB:", collection.count())

Stored in ChromaDB: 507


Retrieval-Augmented Generation (RAG) pipeline

In [30]:
import os, getpass
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception:
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

In [31]:
preferred = [
    "google/gemma-4-31b-it:free",
    "nvidia/nemotron-3-super-120b-a12b:free",
    "google/gemma-4-26b-a4b-it:free",
]

try:
    found = [m.id for m in client.models.list().data if m.id.endswith(":free")]
except Exception:
    found = []

bad = ("guard", "safety", "embed", "moderation")
found = [m for m in found if not any(b in m.lower() for b in bad)]
FREE_MODELS = preferred + [m for m in found if m not in preferred]
print(len(FREE_MODELS), "free models available")

def ask_llm(prompt, system="You are a helpful assistant.", temperature=0.0):
    for m in FREE_MODELS[:12]:
        try:
            r = client.chat.completions.create(
                model=m,
                temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": prompt}],
            )
            text = r.choices[0].message.content
            if text and text.strip() and not text.strip().lower().startswith("user safety"):
                print("Model used:", m)
                return text
        except Exception:
            continue
    return "No model responded."

16 free models available


Take user query

In [32]:
question = "What is the impact of climate change on human health?"

Retrieve the most relevant paragraph using ChromaDB

In [33]:
query_embedding = model.encode([question])
results = collection.query(query_embeddings=query_embedding.tolist(), n_results=1)
context = results["documents"][0][0]
source = results["metadatas"][0][0]["source"]

Generate the answer using the LLM API based on the retrieved text

In [34]:
SYSTEM_PROMPT = (
    "Answer ONLY using the provided context. "
    "Write a clear answer in 2-3 complete sentences. "
    "If the answer is not in the context, say: 'I don't know based on the available documents.'"
)

prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
answer = ask_llm(prompt, system=SYSTEM_PROMPT)

Model used: recursive/flint-v1.0:free


Display the retrieved context and the final LLM-generated answer

In [35]:
print("Question:", question)
print("\nSource Page:", source)
print("\nRetrieved Context:\n", context)
print("\nLLM Generated Answer:\n", answer)

Question: What is the impact of climate change on human health?

Source Page: Effects of climate change

Retrieved Context:
 Humans are vulnerable to climate change in many ways. Sources of food and fresh water can be threatened by environmental changes. Human health can be impacted by weather extremes or by ripple effects like the spread of infectious diseases. Economic impacts include changes to agriculture, fisheries, and forestry. Higher temperatures will increasingly prevent outdoor labor in tropical latitudes due to heat stress. Island nations and coastal cities may be inundated by rising sea levels. Some groups of people may be particularly at risk from climate change, such as the poor, children, and indigenous peoples. Industrialised countries, which have emitted the vast majority of CO2, have more resources to adapt to global warming than developing nations do. Cumulative effects and extreme weather events can lead to displacement and migration.

LLM Generated Answer:
 Accordi